# 03 — Transfer learning

Redes pré-treinadas no ImageNet com entrada redimensionada para 224 px, nas três estratégias (`frozen`, `partial`, `full`). Os hiperparâmetros são os mesmos para todas as configurações deste notebook.

In [ ]:
# Setup: no Colab (site ou extensão do VS Code) clona o repositório do GitHub
# e instala as dependências; localmente só ajusta o sys.path.
# USE_DRIVE guarda pesos, JSONs e figuras no Google Drive para sobreviverem ao
# fim da sessão (o notebook 04 precisa dos pesos).
import os, shutil, sys
USE_DRIVE = True
# Detecta o Colab pela pasta /content em vez de "google.colab": no kernel da
# extensão do VS Code esse módulo pode não estar carregado.
IN_COLAB = os.path.isdir("/content") or "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/deep-vision-classification"
    # O código que roda é o do GitHub, não o da sua máquina: faça push antes.
    if not os.path.exists(REPO):
        !git clone -q https://github.com/davimoljo/deep-vision-classification.git {REPO}
    else:
        !git -C {REPO} pull -q
    %cd {REPO}
    !pip install -q -r requirements.txt
    import torch
    # Sem GPU o treino cai para a CPU sem avisar: a baseline levou ~3 h em vez
    # de ~15 min. Melhor parar aqui do que descobrir no fim do treino.
    if not torch.cuda.is_available():
        raise RuntimeError("Servidor sem GPU. Remova este servidor e crie outro "
                           "do Colab com GPU (T4), depois rode esta célula de novo.")
    print("GPU:", torch.cuda.get_device_name(0))
    if USE_DRIVE:
        if not os.path.isdir("/content/drive/MyDrive"):
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:
                print("Não montou o Drive:", e)
                print("No VS Code use 'Colab: Mount Google Drive to Server...' e rode esta célula de novo.")
        if os.path.isdir("/content/drive/MyDrive"):
            os.environ["OUTPUT_ROOT"] = "/content/drive/MyDrive/deep-vision-classification"
            print("Saídas em", os.environ["OUTPUT_ROOT"])
            # Baixar o CIFAR-10 do site original leva ~15 min a cada servidor
            # novo; com uma cópia do arquivo no Drive, as próximas vezes levam
            # segundos. O torchvision reaproveita o .tar.gz se o MD5 bater.
            cache = os.path.join(os.environ["OUTPUT_ROOT"], "data", "cifar-10-python.tar.gz")
            local = os.path.join(REPO, "data", "cifar-10-python.tar.gz")
            os.makedirs(os.path.dirname(local), exist_ok=True)
            if os.path.exists(cache) and not os.path.exists(local):
                shutil.copy(cache, local)
            from torchvision import datasets
            datasets.CIFAR10(os.path.join(REPO, "data"), train=True, download=True)
            if not os.path.exists(cache) and os.path.exists(local):
                os.makedirs(os.path.dirname(cache), exist_ok=True)
                shutil.copy(local, cache)
        else:
            print("ATENÇÃO: sem Drive, resultados e pesos somem quando o servidor desligar.")
    ROOT = REPO
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)

In [ ]:
from src.pipeline import (DEFAULTS, STRATEGIES, build_cnn, build_from_config,
                          build_pretrained, count_parameters, evaluate_on_test, fit,
                          get_loaders, list_results, load_best_weights, load_result,
                          result_exists)
from src.plots import comparison_table, per_class_accuracy, plot_confusion, plot_curves
DEFAULTS

In [ ]:
IMG_SIZE = 224
# PROPOSTA — confirmar com o grupo.
# Em 224 px cada época custa muito mais que em 32 px; menos épocas é o que
# cabe no tempo de GPU do Colab. O valor é o mesmo para todas as redes.
# Se o lr for alterado aqui (ex.: 1e-4 para não destruir os pesos do ImageNet
# no fine-tuning completo), o ajuste precisa ser justificado no relatório.
TL_TRAIN = {"epochs": 10}
# O enunciado exige no mínimo 2 arquiteturas x 2 estratégias. As 6 x 3 = 18
# runs abaixo dificilmente cabem no Colab até a entrega: escolham 2 ou 3.
ARCHS = ["resnet18", "resnet50", "vgg16", "densenet121", "mobilenet_v2", "efficientnet_b0"]

In [ ]:
loaders = get_loaders(DEFAULTS["batch_size"], img_size=IMG_SIZE, augment=True)
for arch in ARCHS:
    for strategy in STRATEGIES:
        run = f"tl_{arch}_{strategy}"
        if result_exists(run):
            print(f"{run}: já existe, pulando")
            continue
        fit(build_pretrained(arch, strategy), loaders, run, **TL_TRAIN)

## Opcional: mesma arquitetura treinada do zero

Isola o efeito dos pesos do ImageNet.

In [ ]:
RUN_SCRATCH = "tl_resnet18_scratch"
if not result_exists(RUN_SCRATCH):
    fit(build_pretrained("resnet18", "full", pretrained=False), loaders, RUN_SCRATCH, **TL_TRAIN)

## Comparação (validação)

In [ ]:
runs = list_results("tl_")
comparison_table(runs, name="transfer").sort_values("Val acc (%)", ascending=False)

In [ ]:
plot_curves([f"tl_resnet18_{s}" for s in STRATEGIES], name="curvas_resnet18_estrategias");